# Data Preparation — Classifier Service

This notebook cleans and splits the raw labelled dataset into the six split files
consumed by the training notebooks and the held-out eval gate.

**Outputs** (written to `data/cleaned/`):
- `clean_raw_train.csv` / `clean_raw_val.csv` / `clean_raw_test.csv`
- `clean_strict_train.csv` / `clean_strict_val.csv` / `clean_strict_test.csv`

**Metadata** written to `data/cleaning_metadata.json`.

Run once per raw-dataset revision. Pin the dataset revision in `data/data_card.md`.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# Path TBD with teammate — confirm before running.
RAW = Path("data/raw_merged.csv")
OUT = Path("data/cleaned")
OUT.mkdir(parents=True, exist_ok=True)
VALID = {"SPAM", "FAQ", "ACCOUNT_OPS", "HARD_QUESTION", "UNKNOWN"}

df = pd.read_csv(RAW)
metadata = {"raw_rows": len(df), "drops": {}}

before = len(df)
df["message"] = df["message"].astype(str).str.strip()
df = df[df["message"].str.len() > 0]
metadata["drops"]["empty_message"] = before - len(df)

before = len(df)
df = df[df["label"].isin(VALID)]
metadata["drops"]["unknown_label"] = before - len(df)

raw = df.copy()

strict = raw.copy()
strict["message"] = strict["message"].str.lower().apply(lambda s: re.sub(r"\s+", " ", s))
before = len(strict)
strict = strict.drop_duplicates(subset=["message"])
metadata["drops"]["strict_dupes"] = before - len(strict)


def split_and_write(frame, prefix):
    train, temp = train_test_split(frame, test_size=0.2, stratify=frame["label"], random_state=42)
    val, test = train_test_split(temp, test_size=0.5, stratify=temp["label"], random_state=42)
    train.to_csv(OUT / f"{prefix}_train.csv", index=False)
    val.to_csv(OUT / f"{prefix}_val.csv", index=False)
    test.to_csv(OUT / f"{prefix}_test.csv", index=False)
    return {"train": len(train), "val": len(val), "test": len(test)}


metadata["raw_split"] = split_and_write(raw, "clean_raw")
metadata["strict_split"] = split_and_write(strict, "clean_strict")
metadata["label_distribution"] = raw["label"].value_counts().to_dict()
Path("data/cleaning_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True))
metadata
